# 02 — Limpieza y Transformación de Datos
**AgroVision AI** · Pipeline de calidad de datos

## 1. Carga de datos crudos

In [ ]:
import pandas as pd
import numpy as np
import requests
import warnings
warnings.filterwarnings('ignore')

URL = "https://www.datos.gov.co/resource/uejq-wxrr.json"
params = {"$limit": 10000, "$where": "rendimiento IS NOT NULL AND rendimiento > 0", "$order": "a_o DESC"}
df_raw = pd.DataFrame(requests.get(URL, params=params, timeout=60).json())
print(f"Datos crudos: {df_raw.shape}")

## 2. Renombrado de columnas

In [ ]:
rename_map = {
    'a_o': 'anio', 'rea_sembrada': 'area_sembrada',
    'rea_cosechada': 'area_cosechada', 'producci_n': 'produccion',
    'desagregaci_n_cultivo': 'desagregacion_cultivo',
}
df = df_raw.rename(columns={k:v for k,v in rename_map.items() if k in df_raw.columns})

for col in ['area_sembrada','area_cosechada','produccion','rendimiento','anio']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Columnas renombradas: {list(df.columns)}")

## 3. Filtros de calidad

In [ ]:
n_inicial = len(df)

# Filtro 1: rendimiento válido
df = df[df['rendimiento'] > 0]
print(f"Después de rendimiento > 0: {len(df)} (-{n_inicial - len(df)})")

# Filtro 2: outliers agronómicos
n2 = len(df)
df = df[df['rendimiento'] < 50]
print(f"Después de rendimiento < 50: {len(df)} (-{n2 - len(df)})")

# Filtro 3: área sembrada válida
n3 = len(df)
df = df.dropna(subset=['area_sembrada'])
df = df[df['area_sembrada'] > 0]
print(f"Después de area_sembrada válida: {len(df)} (-{n3 - len(df)})")

print(f"\nTotal registros eliminados: {n_inicial - len(df)} ({(n_inicial-len(df))/n_inicial*100:.1f}%)")

## 4. Deduplicación

In [ ]:
n_antes = len(df)
cols_clave = [c for c in ['municipio','cultivo','anio','periodo_num','desagregacion_cultivo'] if c in df.columns]
if 'periodo' in df.columns:
    df['periodo_num'] = df['periodo'].apply(lambda x: 1 if 'primer' in str(x).lower() else (2 if 'segundo' in str(x).lower() else 0))
elif 'periodo_num' not in df.columns:
    df['periodo_num'] = 0

clave = [c for c in ['municipio','cultivo','anio','periodo_num'] if c in df.columns]
df = df.sort_values('rendimiento', ascending=True).drop_duplicates(subset=clave, keep='last').reset_index(drop=True)

print(f"Registros antes: {n_antes}")
print(f"Registros después (deduplicados): {len(df)}")
print(f"Duplicados eliminados: {n_antes - len(df)}")

## 5. Imputación selectiva

In [ ]:
# area_cosechada nula → usar area_sembrada como proxy
if 'area_cosechada' in df.columns:
    nulos_ac = df['area_cosechada'].isnull().sum()
    df['area_cosechada'] = df['area_cosechada'].fillna(df['area_sembrada'])
    print(f"area_cosechada: {nulos_ac} nulos imputados con area_sembrada")

# Variables categóricas → DESCONOCIDO
for col in ['municipio','departamento','cultivo','grupo_cultivo']:
    if col in df.columns:
        df[col] = df[col].fillna('DESCONOCIDO').str.upper().str.strip()

print("\nNulos restantes por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 6. Variables derivadas

In [ ]:
# ratio_cosecha
df['ratio_cosecha'] = (df['area_cosechada'] / df['area_sembrada'].replace(0, np.nan)).clip(0, 1).fillna(0)

# rendimiento_hist_prom
df['rendimiento_hist_prom'] = df.groupby(['departamento','cultivo'])['rendimiento'].transform('mean')

# ONI index por año (tabla histórica NOAA)
oni_map = {2019: 0.5, 2020: -1.2, 2021: -1.0, 2022: -0.9, 2023: 2.0, 2024: -0.6}
df['oni_index'] = df['anio'].map(oni_map).fillna(0.0)

# riesgo_climatico
def cat_oni(v):
    if v <= -1.5: return 0
    if v <= -0.5: return 1
    if v <   0.5: return 2
    if v <   1.5: return 3
    return 4
df['riesgo_climatico_enc'] = df['oni_index'].apply(cat_oni)

print("Variables derivadas creadas:")
print(df[['ratio_cosecha','rendimiento_hist_prom','oni_index','riesgo_climatico_enc']].describe())

## 7. Dataset limpio final

In [ ]:
print(f"Shape final: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.to_csv('data_limpio.csv', index=False)
print("\nGuardado en data_limpio.csv")
df.head()